In [1]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.utils import simple_preprocess

print("Loading saved model and artifacts...")
print("=" * 80)

loaded_model = load_model('lstm_intent_classifier_9kdataset.keras')
print("✓ Model loaded from 'lstm_intent_classifier_9kdataset.keras'")

with open('tokenizer_9kdataset.pkl', 'rb') as f:
    loaded_tokenizer = pickle.load(f)
print("✓ Tokenizer loaded from 'tokenizer_9kdataset.pkl'")

with open('model_config_9kdataset.pkl', 'rb') as f:
    loaded_config = pickle.load(f)
print("✓ Configuration loaded from 'model_config_9kdataset.pkl'")

print("\nModel Configuration:")
print(f"  Max sequence length: {loaded_config['max_length']}")
print(f"  Vocabulary size: {loaded_config['vocab_size']}")
print(f"  Embedding dimension: {loaded_config['embedding_dim']}")
print("=" * 80)

/Users/geetansh/Desktop/verve/chatbot/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Loading saved model and artifacts...
✓ Model loaded from 'lstm_intent_classifier_9kdataset.keras'
✓ Tokenizer loaded from 'tokenizer_9kdataset.pkl'
✓ Configuration loaded from 'model_config_9kdataset.pkl'

Model Configuration:
  Max sequence length: 33
  Vocabulary size: 2385
  Embedding dimension: 300


In [2]:
def predict_intent_from_loaded_model(sentence):
    def preprocess_text(text):
        tokens = simple_preprocess(text, deacc=True)
        return tokens
    
    tokens = preprocess_text(sentence)
    preprocessed_text = ' '.join(tokens)
    
    with open('tokenizer_9kdataset.pkl', 'rb') as f:
        loaded_tokenizer = pickle.load(f)

    sequence = loaded_tokenizer.texts_to_sequences([preprocessed_text])
    
    padded = pad_sequences(
        sequence, 
        maxlen=loaded_config['max_length'], 
        padding='post', 
        truncating='post'
    )
    
    prediction = loaded_model.predict(padded, verbose=0)
    predicted_class = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class]
    
    reverse_label_mapping = {0: -2, 1: -1, 2: 1, 3: 2}
    original_label = reverse_label_mapping[predicted_class]
    
    label_descriptions = {
        -2: "Escalate to human agent",
        -1: "Track order/order status (login required)",
        1: "General conversation/company policy",
        2: "Product recommendation"
    }
    
    return {
        'sentence': sentence,
        'preprocessed': tokens,
        'label': original_label,
        'description': label_descriptions[original_label],
        'confidence': float(confidence),
        'all_probabilities': {
            '-2 (Escalate)': float(prediction[0][0]),
            '-1 (Track order)': float(prediction[0][1]),
            '1 (Conversation)': float(prediction[0][2]),
            '2 (Recommendation)': float(prediction[0][3])
        }
    }

print("✓ Inference function ready!")
print("\nUsage: predict_intent_from_loaded_model('your sentence here')")

✓ Inference function ready!

Usage: predict_intent_from_loaded_model('your sentence here')


In [3]:
while True:
    user_input = input("\nEnter a sentence to classify (or 'exit' to quit): ")
    if user_input.lower() == 'exit':
        print("Exiting inference loop.")
        break
    prediction = predict_intent_from_loaded_model(user_input)
    print(f"\nPredicted Label: {prediction['label']} - {prediction['description']}")
    print(f"Confidence: {prediction['confidence']:.2%}")
    print("Probabilities:")
    for label, prob in prediction['all_probabilities'].items():
        print(f"  {label}: {prob:.4f}")


Predicted Label: 2 - Product recommendation
Confidence: 81.66%
Probabilities:
  -2 (Escalate): 0.0082
  -1 (Track order): 0.0140
  1 (Conversation): 0.1612
  2 (Recommendation): 0.8166

Predicted Label: -2 - Escalate to human agent
Confidence: 99.99%
Probabilities:
  -2 (Escalate): 0.9999
  -1 (Track order): 0.0000
  1 (Conversation): 0.0001
  2 (Recommendation): 0.0000

Predicted Label: 2 - Product recommendation
Confidence: 51.96%
Probabilities:
  -2 (Escalate): 0.0300
  -1 (Track order): 0.0530
  1 (Conversation): 0.3974
  2 (Recommendation): 0.5196

Predicted Label: -1 - Track order/order status (login required)
Confidence: 99.97%
Probabilities:
  -2 (Escalate): 0.0001
  -1 (Track order): 0.9997
  1 (Conversation): 0.0002
  2 (Recommendation): 0.0000
Exiting inference loop.
